In [47]:
import pandas as pd
import requests
from bs4 import BeautifulSoup as be
import lxml
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [57]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Referer": "https://www.ambitionbox.com/"
})
def fetch_page(page):
    url = f"https://www.ambitionbox.com/list-of-companies?page={page}"
    r = session.get(url, timeout=20)
    if r.status_code == 200 and "Page not found" not in r.text:
        return r.text
    
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1280,800")
    driver = webdriver.Chrome(options=opts)
    driver.get(url)
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "h2.companyCardWrapper__companyName"))
    )
    html = driver.page_source
    driver.quit()
    return html

all_pages = {}
for page in range(1, 501):
    try:
        print(f"Fetching page {page}...")
        html = fetch_page(page)
        all_pages[page] = html
    except Exception as e:
        print(f"Error fetching page {page}: {e}")

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...
Fetching page 6...
Fetching page 7...
Fetching page 8...
Fetching page 9...
Fetching page 10...
Fetching page 11...
Fetching page 12...
Fetching page 13...
Fetching page 14...
Fetching page 15...
Fetching page 16...
Fetching page 17...
Fetching page 18...
Fetching page 19...
Fetching page 20...
Fetching page 21...
Fetching page 22...
Fetching page 23...
Fetching page 24...
Fetching page 25...
Fetching page 26...
Fetching page 27...
Fetching page 28...
Fetching page 29...
Fetching page 30...
Fetching page 31...
Fetching page 32...
Fetching page 33...
Fetching page 34...
Fetching page 35...
Fetching page 36...
Fetching page 37...
Fetching page 38...
Fetching page 39...
Fetching page 40...
Fetching page 41...
Fetching page 42...
Fetching page 43...
Fetching page 44...
Fetching page 45...
Fetching page 46...
Fetching page 47...
Fetching page 48...
Fetching page 49...
Fetching page 50...
Fetching 

In [58]:
webpage = all_pages.get(1, "")
soup = be(webpage, "lxml")
print(soup.prettify()[:2000])

<!DOCTYPE html>
<html data-n-head="%7B%22lang%22:%7B%22ssr%22:%22en%22%7D%7D" data-n-head-ssr="" lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width,initial-scale=1,minimum-scale=1" name="viewport"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <link href="/assets/next/manifest.json" rel="manifest"/>
  <style>
   @media only screen and (min-width:767px){.trp-img{width:400px!important;max-width:400px!important}}
  </style>
  <script defer="" src="/static/js/env-runtime.js">
  </script>
  <script>
   window.dataLayer=window.dataLayer||[],window.gtag=window.gtag||function(){window.dataLayer.push(arguments)},gtag("js",new Date),window.initialDate=(new Date).toISOString()
  </script>
  <script>
   window.Prism=window.Prism||{},window.Prism.manual=!0
  </script>
  <title>
   Top Companies in India | AmbitionBox
  </title>
  <meta content="2026 AmbitionBox" data-n-head="ssr" name="copyright"/>
  <meta content="1 day" data-n-head="ssr" name="revisit-a

In [59]:
webpage = all_pages.get(1, "")
print(webpage[:2000])  # Run this to see what you're actually getting

<!doctype html>
<html data-n-head-ssr lang="en" data-n-head="%7B%22lang%22:%7B%22ssr%22:%22en%22%7D%7D">
  <head >
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width,initial-scale=1,minimum-scale=1">
    <meta http-equiv="X-UA-Compatible" content="IE=edge"> 
    <link rel="manifest" href="/assets/next/manifest.json">
    <style>@media only screen and (min-width:767px){.trp-img{width:400px!important;max-width:400px!important}}</style>
    <script src="/static/js/env-runtime.js" defer></script>
    <script>window.dataLayer=window.dataLayer||[],window.gtag=window.gtag||function(){window.dataLayer.push(arguments)},gtag("js",new Date),window.initialDate=(new Date).toISOString()</script>
    <script>window.Prism=window.Prism||{},window.Prism.manual=!0</script>
    <title>Top Companies in India | AmbitionBox</title><meta data-n-head="ssr" name="copyright" content="2026 AmbitionBox"><meta data-n-head="ssr" name="revisit-after" content="1 day"><meta data-n-head="ss

In [60]:
soups = {page: be(html, "lxml") for page, html in all_pages.items()}
rows = []
for page, soup in soups.items():
    for card in soup.select("div.companyCardWrapper"):
        name_el = card.select_one("h2.companyCardWrapper__companyName")
        rating_el = card.select_one("div.rating_text.rating_text--md")
        good_hdr = card.select_one("span.companyCardWrapper__ratingHeader--high")
        good_val = good_hdr.find_next("span", class_="companyCardWrapper__ratingValues") if good_hdr else None
        bad_hdr = card.select_one("span.companyCardWrapper__ratingHeader--critical")
        bad_val = bad_hdr.find_next("span", class_="companyCardWrapper__ratingValues") if bad_hdr else None
        salary_count = None
        for a in card.select("a.companyCardWrapper__ActionWrapper"):
            title = a.select_one("span.companyCardWrapper__ActionTitle")
            count = a.select_one("span.companyCardWrapper__ActionCount")
            if title and "Salaries" in title.get_text(strip=True):
                salary_count = count.get_text(strip=True) if count else None
                break
        if name_el and rating_el:
            rows.append({
                "Company Name": name_el.get_text(strip=True),
                "Rating": rating_el.get_text(strip=True),
                "Good Reviews": (good_val.get_text(strip=True) if good_val else None),
                "Bad Reviews": (bad_val.get_text(strip=True) if bad_val else None),
                "Salaries": salary_count
            })
print(f"Collected {len(rows)} rows")

Collected 9998 rows


In [61]:
# parsing is handled in the previous cell; this cell is now informational
print("Parsing completed across pages")

Parsing completed across pages


In [62]:
final_result_df = pd.DataFrame(rows)
final_result_df.to_csv("company_data.csv", index=False)
print("Saved company_data.csv", len(final_result_df))

Saved company_data.csv 9998
